# Experiment 4 — Multi-Start ICP: Effect of `n_starts`

**Goal:** Examine how the number of random starting rotations (`n_starts`) affects ICP alignment quality.

For each value of `n_starts` ∈ [1, 20] we run `MultiStartICP` over multiple random experiments and measure:
- Rotation error (°) vs. ground truth
- Translation error vs. ground truth
- Mean point residual of the best trial

The shaded band shows ± 1 std across seeds.

In [2]:
import sys
sys.path.insert(0, '../src')

import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from icp import ICP, MultiStartICP, MultiSeedSyntheticICPResult, fit_multi_seed
from visualization import ErrorMetricsVisualizer

plt.style.use('seaborn-v0_8-whitegrid')

## 1. Setup

We use a **clustered** point cloud (8 Gaussian blobs), which has enough geometric structure to create meaningful local minima — the scenario where multi-start ICP is most useful.

In [3]:
N_SEEDS = 20
N_STARTS_RANGE = list(range(1, 21))  # 1 … 20

# Fixed ICP configuration reused across all trials
BASE_ICP = ICP(max_iter=200, tol=1e-10, verbose=False)

## 2. Data collection

For every `n_starts` value we run `fit_multi_seed`

In [ ]:
results: list[MultiSeedSyntheticICPResult] = []
for i, n_starts in enumerate(tqdm(N_STARTS_RANGE, desc='n_starts')):
    multi_icp = MultiStartICP(BASE_ICP, n_starts=n_starts, seed=0)
    experiment_kwargs = {
        "n": 500, "noise_std": 0.01, "t_scale": 8.0, "style": 'clustered'
    }
    multi_icp_results = fit_multi_seed(multi_icp, list(range(N_SEEDS)), experiment_kwargs=experiment_kwargs, verbose=False)
    results.append(multi_icp_results)

n_starts:  70%|███████   | 14/20 [06:07<04:12, 42.05s/it]

## 3. Results

Mean ± 1 std over 20 random experiments for each metric.

In [ ]:
rot_errors = np.array([r.rotation_errors for r in results])
t_errors = np.array([r.translation_errors for r in results])
residuals = np.array([r.mean_residuals for r in results])

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
ErrorMetricsVisualizer.plot_parameter_sweep(
    axes,
    x=N_STARTS_RANGE,
    data_per_metric=[rot_errors, t_errors, residuals],
    y_labels=['Rotation error (°)', 'Translation error', 'Mean point residual'],
    x_label='n_starts',
)

plt.suptitle(
    f'Multi-Start ICP — effect of n_starts  ({N_SEEDS} seeds per value)',
    y=1.02
)
plt.tight_layout()
plt.savefig('../results/6_multi_start_icp_error_vs_n_starts.png', dpi=150, bbox_inches='tight')
plt.show()